## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [2]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
#!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
#!CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python --force-reinstall --no-cache-dir -q

!CMAKE_ARGS="-DLLAMA_METAL=on" pip install --force-reinstall --no-cache-dir llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 51.7 MB/s  0:00:00 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 53.6 MB/s  0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.16-cp313-cp313-macosx_26_0_arm64.whl size=3842966 sha256=cea0d92fa77a9163e2fb8daa00a70b86687f38e6f7e16ece43706460cd8c663c
  Stored in directory: /private/var/folders/qx/cd3rb17s6q93jfnkvf109f6r0000gn/T/pip-ephem-wheel-cache-u7etopek/wheels/af/3a/b6/445d9f4ccadd3ed923d55af8f055f2ccd217c66f09c834f0d8
Successfully built llama-cpp-python
  Attempting uninstall: typing-extensions
    Found existing installation: typing_extensions 4.15.0
    Uninstalling typing_extensions-4.15.0:
      Successfully uninstalled typing_extensions-4.15.032m0/6 [typing-extensions]
  Attempting uninstall: numpy━━━━━━━━━━━━━━━ 0/6 [typing-extensions]
   

In [3]:
# For installing the libraries & downloading models from HF Hub
%pip install huggingface_hub pandas tiktoken pymupdf langchain langchain-community chromadb sentence-transformers numpy -q

Note: you may need to restart the kernel to use updated packages.


In [4]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

In [5]:
#mistral-7b-instruct-v0.2.Q5_K_S.gguf suggested by https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF
model_path = hf_hub_download(repo_id="TheBloke/Mistral-7B-Instruct-v0.2-GGUF", filename="mistral-7b-instruct-v0.2.Q5_K_S.gguf")

In [6]:
llm = Llama(
    model_path=model_path, 
    n_ctx=2048, 
    n_threads=8, 
    n_gpu_layers=35, 
    temperature=0.7,
    repeat_penalty=1.1)

llama_model_load_from_file_impl: using device Metal (Apple M1 Max) - 25558 MiB free
llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /Users/parvatam/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q5_K_S.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader:

#### Response

In [12]:
from IPython.display import Markdown, display

def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return Markdown(model_output['choices'][0]['text'])

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [13]:
sepsis_query1="What is the protocol for managing sepsis in a critical care unit?"
response(sepsis_query1)
# checking llm response

Llama.generate: 2 prefix-match hit, remaining 14 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =     375.91 ms /    14 tokens (   26.85 ms per token,    37.24 tokens per second)
llama_perf_context_print:        eval time =    2821.39 ms /   127 runs   (   22.22 ms per token,    45.01 tokens per second)
llama_perf_context_print:       total time =    3255.41 ms /   141 tokens
llama_perf_context_print:    graphs reused =        122




Sepsis is a life-threatening condition that can arise from an infection, and it is important to recognize and manage it promptly in a critical care unit. The following steps outline the general protocol for managing sepsis in a critical care unit:

1. Early recognition: Recognize the signs and symptoms of sepsis, which may include fever, chills, rapid heart rate, rapid breathing, confusion, and low blood pressure. Suspect sepsis in any patient with suspected or confirmed infection who is showing signs of organ dysfunction.
2. Resuscitation: Begin resusc

##### above code output shows llm is responding with some data

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [14]:

appendicitis_query2="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

response(appendicitis_query2)

# checking llm response for query2

Llama.generate: 2 prefix-match hit, remaining 32 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =     255.00 ms /    32 tokens (    7.97 ms per token,   125.49 tokens per second)
llama_perf_context_print:        eval time =    2770.81 ms /   127 runs   (   21.82 ms per token,    45.83 tokens per second)
llama_perf_context_print:       total time =    3069.74 ms /   159 tokens
llama_perf_context_print:    graphs reused =        122




Appendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The symptoms of appendicitis can vary from person to person, but the following are the most common:

1. Abdominal pain: The pain is usually sharp, constant, and localized in the lower right abdomen. It may start as a mild discomfort, but it can quickly worsen and become severe.
2. Loss of appetite: People with appendicitis may lose their appetite and feel nauseous.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [15]:
hairLoss_query3="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

response(hairLoss_query3)

Llama.generate: 4 prefix-match hit, remaining 34 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =     384.00 ms /    34 tokens (   11.29 ms per token,    88.54 tokens per second)
llama_perf_context_print:        eval time =    2804.02 ms /   127 runs   (   22.08 ms per token,    45.29 tokens per second)
llama_perf_context_print:       total time =    3245.50 ms /   161 tokens
llama_perf_context_print:    graphs reused =        122




Sudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles, leading to hair loss in small, round patches on the scalp, beard, or other areas of the body. The exact cause of alopecia areata is not known, but it is believed to be related to a combination of genetic and environmental factors.

There are several treatments and solutions for addressing sudden patchy hair loss:

1. Topical corticosteroids: These are anti-inflammatory medications that can be applied

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [16]:
brainTissue_query4="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"

response(brainTissue_query4)

Llama.generate: 2 prefix-match hit, remaining 28 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =     283.69 ms /    28 tokens (   10.13 ms per token,    98.70 tokens per second)
llama_perf_context_print:        eval time =    2821.15 ms /   127 runs   (   22.21 ms per token,    45.02 tokens per second)
llama_perf_context_print:       total time =    3173.33 ms /   155 tokens
llama_perf_context_print:    graphs reused =        122




There is no one-size-fits-all answer to this question, as the specific treatment recommendations for a person with a brain injury depend on the severity and location of the injury, as well as the individual's overall health and medical history. However, I can provide some general information about common treatments and interventions that may be used to help manage the symptoms and promote recovery after a brain injury.

1. Medical management: Depending on the severity of the injury, the person may require hospitalization and intensive care to manage life-threatening conditions such as brain swelling, bleeding,

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [17]:
legFracture_query5="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"

response(legFracture_query5)

Llama.generate: 2 prefix-match hit, remaining 35 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =     816.26 ms /    35 tokens (   23.32 ms per token,    42.88 tokens per second)
llama_perf_context_print:        eval time =    2868.41 ms /   127 runs   (   22.59 ms per token,    44.28 tokens per second)
llama_perf_context_print:       total time =    3754.78 ms /   162 tokens
llama_perf_context_print:    graphs reused =        122




First and foremost, if you suspect that someone has fractured their leg during a hiking trip, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:

1. Keep the person calm and still: Encourage the person to remain calm and avoid moving the injured leg as much as possible to prevent further damage or discomfort.
2. Assess the injury: Check the leg for signs of deformity, swelling, or bruising. If the person is unable to put weight on the leg or is experiencing severe pain, it's likely that

## Question Answering using LLM with Prompt Engineering

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [18]:
sepsis_prompt="Answer this question as a medical researcher to help other researchers."
response(sepsis_prompt+sepsis_query1)

Llama.generate: 1 prefix-match hit, remaining 28 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =     423.87 ms /    28 tokens (   15.14 ms per token,    66.06 tokens per second)
llama_perf_context_print:        eval time =    2855.90 ms /   127 runs   (   22.49 ms per token,    44.47 tokens per second)
llama_perf_context_print:       total time =    3338.06 ms /   155 tokens
llama_perf_context_print:    graphs reused =        122




As a medical researcher, I would recommend the following protocol for managing sepsis in a critical care unit, based on current evidence-based guidelines and best practices:

1. Early recognition and diagnosis: Sepsis should be suspected in any patient with suspected or confirmed infection and signs of organ dysfunction. Use the Sequential Organ Failure Assessment (SOFA) score or Quick Sequential Organ Failure Assessment (qSOFA) score to identify patients at risk for sepsis.
2. Fluid resuscitation: Aggressively resuscitate patients with intravenous fluids

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [21]:
appendicitis_prompt="Answer this question as a expert medical surgeon to another surgeon who already know about appendicitis."

response(appendicitis_prompt+appendicitis_query2)

Llama.generate: 14 prefix-match hit, remaining 41 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =     531.06 ms /    41 tokens (   12.95 ms per token,    77.20 tokens per second)
llama_perf_context_print:        eval time =    2858.04 ms /   127 runs   (   22.50 ms per token,    44.44 tokens per second)
llama_perf_context_print:       total time =    3462.01 ms /   168 tokens
llama_perf_context_print:    graphs reused =        122




As a medical surgeon, I would explain that the common symptoms for appendicitis include:

1. Periumbilical or right lower quadrant abdominal pain that begins as a vague discomfort and progresses to sharp, localized pain.
2. Anorexia and nausea, which may lead to vomiting.
3. Low-grade fever, often below 101°F (38.3°C).
4. Loss of appetite.
5. Rebound tenderness, which is pain when the abdomen is pressed gently and then released.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [22]:
hairLoss_prompt="Answer this question as a dermatologist to another dermatologist concisely."
response(hairLoss_prompt+hairLoss_query3)

Llama.generate: 6 prefix-match hit, remaining 50 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =    1252.68 ms /    50 tokens (   25.05 ms per token,    39.91 tokens per second)
llama_perf_context_print:        eval time =    2862.90 ms /   127 runs   (   22.54 ms per token,    44.36 tokens per second)
llama_perf_context_print:       total time =    4197.74 ms /   177 tokens
llama_perf_context_print:    graphs reused =        122




As a dermatologist, I would suggest the following potential causes and treatments for sudden, patchy hair loss:

1. Alopecia Areata: An autoimmune disorder that causes hair loss in circular patches. Treatment options include topical corticosteroids, intralesional triamcinolone, systemic corticosteroids, immunomodulators, and phototherapy.
2. Tinea Capitis: A fungal infection of the scalp that can cause patchy hair loss. Treatment involves antifungal shampoos and oral antifung

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [24]:
brainTissue_prompt="Answer this question as a expert neurologist to a PHD student concisely."

response(brainTissue_prompt+brainTissue_query4)

Llama.generate: 15 prefix-match hit, remaining 32 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =     273.37 ms /    32 tokens (    8.54 ms per token,   117.06 tokens per second)
llama_perf_context_print:        eval time =    2783.76 ms /   127 runs   (   21.92 ms per token,    45.62 tokens per second)
llama_perf_context_print:       total time =    3111.31 ms /   159 tokens
llama_perf_context_print:    graphs reused =        122




As a neurology expert, I would recommend the following treatments for a person with a brain injury, depending on the severity and specific symptoms:

1. Acute care: For severe brain injuries, immediate care includes managing airway, breathing, and circulation, controlling intracranial pressure, and preventing or treating seizures.
2. Rehabilitation: Physical, occupational, and speech therapy can help improve motor function, strength, coordination, and communication skills.
3. Medications: Depending on the symptoms, medications may be prescribed to manage seizures, control pain, improve cognitive

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [26]:
legFracture_prompt="Answer this question as a sports specialist to a Athelete in less words"
response(legFracture_prompt+legFracture_query5)

Llama.generate: 13 prefix-match hit, remaining 39 prompt tokens to eval
llama_perf_context_print:        load time =     402.56 ms
llama_perf_context_print: prompt eval time =     419.44 ms /    39 tokens (   10.75 ms per token,    92.98 tokens per second)
llama_perf_context_print:        eval time =    2784.02 ms /   127 runs   (   21.92 ms per token,    45.62 tokens per second)
llama_perf_context_print:       total time =    3263.55 ms /   166 tokens
llama_perf_context_print:    graphs reused =        122




1. Immediate care: R.I.C.E. - Rest, Ice, Compression, Elevation.
2. Seek medical attention: Assess the severity and potential complications.
3. Immobilize the leg: Use a splint or cast to prevent movement.
4. Pain management: Over-the-counter pain relievers or prescription medication.
5. Follow-up care: Regular check-ups with a healthcare professional.
6. Rehabilitation: Physical therapy to regain strength and mobility.
7. Nutrition and hydration: Proper

## Data Preparation for RAG

### Loading the Data

### Data Overview

#### Checking the first 5 pages

#### Checking the number of pages

### Data Chunking

### Embedding

### Vector Database

### Retriever

### System and User Prompt Template

### Response Function

In [ ]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

### Fine-tuning

## Output Evaluation

In [ ]:
groundedness_rater_system_message  = ""

In [ ]:
relevance_rater_system_message = ""

In [ ]:
user_message_template = ""

In [ ]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

## Actionable Insights and Business Recommendations

<font size=6 color='blue'>Power Ahead</font>
___